# Fine-tune จับทรงกล่องจากไดไลน์ v2 (Colab GPU)

**อัปเกรด:** EfficientNet-**B3** + ข้อมูล 500/ทรง + **eval บน expert gold holdout** (วัด accuracy จริง ไม่ใช่ proxy)

**⚠️ สำคัญ:** train ใช้ proxy label (noisy) แต่**วัดผลบน 9 expert-gold** (ตัด family ออกจาก train แล้ว = ไม่ leak)
- proxy เดิมผิด ~33% บนเคส custom → gold บอก accuracy จริง
- baseline: CLIP-kNN 43%, fine-tune B0(200/ทรง)=51%/tuck57%

**ขั้นตอน:** Runtime→GPU(T4) → Run all → อัป `dieline_train.zip`


In [ ]:
!pip -q install timm scikit-learn


### 1. อัปโหลด dieline_train.zip


In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()
zipfile.ZipFile(list(up.keys())[0]).extractall('data')
print('train:', len(os.listdir('data/img')), '| gold:', len(os.listdir('data/gold')))


### 2. โหลด labels + group-split (train/val ภายใน)


In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
df = pd.read_csv('data/labels.csv'); df['y']=df['label']-1
gold = pd.read_csv('data/gold_labels.csv'); gold['y']=gold['label']-1
tr,va = next(GroupShuffleSplit(1,test_size=0.15,random_state=42).split(df,groups=df['group_id']))
train_df,val_df = df.iloc[tr].reset_index(drop=True), df.iloc[va].reset_index(drop=True)
assert not (set(train_df.group_id)&set(val_df.group_id))
print('train',len(train_df),'| val',len(val_df),'| gold(expert)',len(gold))
print(train_df.label.value_counts().sort_index().to_dict())


### 3. Dataset


In [ ]:
import torch, timm, numpy as np, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
SZ=320
aug=T.Compose([T.Resize((SZ,SZ)),T.RandomRotation(4),T.ColorJitter(0.1,0.1),T.ToTensor(),T.Normalize([0.5]*3,[0.5]*3)])
plain=T.Compose([T.Resize((SZ,SZ)),T.ToTensor(),T.Normalize([0.5]*3,[0.5]*3)])
class DS(Dataset):
    def __init__(s,d,folder,tf): s.d=d; s.f=folder; s.tf=tf
    def __len__(s): return len(s.d)
    def __getitem__(s,i):
        r=s.d.iloc[i]; im=Image.open(f'data/{s.f}/{r.file}').convert('RGB')
        return s.tf(im), int(r.y)
tl=DataLoader(DS(train_df,'img',aug),batch_size=24,shuffle=True,num_workers=2)
vl=DataLoader(DS(val_df,'img',plain),batch_size=48,num_workers=2)
gl=DataLoader(DS(gold,'gold',plain),batch_size=16)


### 4. เทรน (EfficientNet-B3, class-weighted, 20 epoch)


In [ ]:
dev='cuda' if torch.cuda.is_available() else 'cpu'; print(dev)
model=timm.create_model('efficientnet_b3',pretrained=True,num_classes=12).to(dev)
cnt=train_df.y.value_counts().sort_index()
w=torch.tensor([len(train_df)/(12*cnt.get(i,1)) for i in range(12)],dtype=torch.float).to(dev)
crit=nn.CrossEntropyLoss(weight=w,label_smoothing=0.05)
opt=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4)
sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=20)
for ep in range(20):
    model.train(); tot=0
    for x,y in tl:
        x,y=x.to(dev),y.to(dev); opt.zero_grad(); loss=crit(model(x),y); loss.backward(); opt.step(); tot+=loss.item()
    sch.step(); model.eval(); c=n=0
    with torch.no_grad():
        for x,y in vl: c+=(model(x.to(dev)).argmax(1).cpu()==y).sum().item(); n+=len(y)
    print('ep%2d loss%.3f val_acc(proxy) %.1f%%'%(ep+1,tot/len(tl),100*c/n))


### 5. ประเมินบน proxy val (เทียบ baseline)


In [ ]:
from sklearn.metrics import classification_report
model.eval(); P=[]; Y=[]
with torch.no_grad():
    for x,y in vl: P+=(model(x.to(dev)).argmax(1).cpu()+1).tolist(); Y+=(y+1).tolist()
P,Y=np.array(P),np.array(Y)
print('=== บน proxy val (noisy) ===')
print('12-way: %.0f%% (CLIP-kNN 43%%, B0 51%%)'%(100*(P==Y).mean()))
tk=np.isin(Y,[1,2,4,11]); print('tuck-family: %.0f%%'%(100*(P[tk]==Y[tk]).mean()))
print('custom-vs-std: %.0f%%'%(100*((P==12)==(Y==12)).mean()))


### 6. 🎯 ประเมินบน EXPERT GOLD (accuracy จริง!)


In [ ]:
model.eval(); GP=[]; GY=[]
with torch.no_grad():
    for x,y in gl: GP+=(model(x.to(dev)).argmax(1).cpu()+1).tolist(); GY+=(y+1).tolist()
GP,GY=np.array(GP),np.array(GY)
print('=== บน EXPERT GOLD (%d ใบ) — accuracy จริง ==='%len(GY))
print('  โมเดล: %.0f%%  (proxy engineer บน gold เดียวกัน = 66%%, Claude/CV = 44%%)'%(100*(GP==GY).mean()))
print('  ต่อใบ (expert vs โมเดลทาย):')
for i in range(len(GY)): print('    expert %2d -> ทาย %2d %s'%(GY[i],GP[i],'✓' if GP[i]==GY[i] else '✗'))
print('  >> ถ้าโมเดลตี custom(12) ถูกในเคสที่ proxy ผิด = โมเดลเก่งกว่า label!')


### 7. เซฟโมเดล


In [ ]:
torch.save(model.state_dict(),'dieline_effb3.pt')
from google.colab import files; files.download('dieline_effb3.pt')
